# Product Taxonomy Classification: 7-Feature Probability Model

This notebook retrains and evaluates the multi-target product taxonomy classification model using an expanded **7-feature pipeline** across all 5 Laerdal taxonomy levels:
- `ProductType`
- `ProductGroup`
- `ProductFamily`
- `ProductLine`
- `ProductKey`

### Feature Pipeline:
1. **Categorical Features (5)**: `GPCBrickCode`, `UNSPSCNumber`, `IsMedical`, `BrandName`, `SubBrandName` (One-Hot Encoded on training split with `handle_unknown='ignore'`)
2. **Dense Semantic Embeddings (2)**: `ProductDescription` (384 dims) and `ProductShortName` (384 dims) encoded with sentence-transformers `all-MiniLM-L6-v2` (768 total dense text dimensions)
3. **Excluded Columns (Leakage Protection)**: `ProductCategory` and `ProductPartType` (directly synonymous with `ProductType`)

### Governance & Tracking:
- Linked Jira Epics & Subtasks: [LBVOICESER-1516](https://laerdal.atlassian.net/browse/LBVOICESER-1516), [LBVOICESER-1564](https://laerdal.atlassian.net/browse/LBVOICESER-1564), [LBVOICESER-1567](https://laerdal.atlassian.net/browse/LBVOICESER-1567)
- Governance SOP & Version Log: `MODEL_CARD.md`, `FEATURE_LOG.md`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
import joblib
import seaborn as sns
from pathlib import Path

RANDOM_STATE = 42

# --- palette: one dict, referenced by role/slot instead of one variable per color ---
PALETTE = {
    "surface":     "#fcfcfb",
    "ink_primary": "#0b0b0b",
    "ink_secondary": "#52514e",
    "ink_muted":   "#898781",
    "gridline":    "#e1e0d9",
    "baseline":    "#c3c2b7",
    "categorical": ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                     "#e87ba4", "#008300", "#4a3aa7", "#e34948"],
    "sequential_blue": ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5",
                         "#256abf", "#184f95", "#0d366b"],
}
blue_cmap = LinearSegmentedColormap.from_list("blue_seq", PALETTE["sequential_blue"])

plt.rcParams.update({
    "figure.facecolor": PALETTE["surface"],
    "axes.facecolor": PALETTE["surface"],
    "axes.edgecolor": PALETTE["baseline"],
    "axes.labelcolor": PALETTE["ink_secondary"],
    "text.color": PALETTE["ink_primary"],
    "xtick.color": PALETTE["ink_muted"],
    "ytick.color": PALETTE["ink_muted"],
    "grid.color": PALETTE["gridline"],
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 1. Load Data & Define Feature Specification

We load the dataset from `datasets/product_classified_Full.xlsx` and specify:
- 5 Categorical features: `GPCBrickCode`, `UNSPSCNumber`, `IsMedical`, `BrandName`, `SubBrandName`
- 2 Text features: `ProductDescription`, `ProductShortName`
- 5 Target hierarchies: `ProductType`, `ProductGroup`, `ProductFamily`, `ProductLine`, `ProductKey`

In [ ]:
DATA_PATH = Path("datasets/product_classified_Full.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("datasets/product_classified_Full.csv")

CAT_FEATURES = ["GPCBrickCode", "UNSPSCNumber", "IsMedical", "BrandName", "SubBrandName"]
TEXT_FEATURES = ["ProductDescription", "ProductShortName"]
TARGETS = ["ProductType", "ProductGroup", "ProductFamily", "ProductLine", "ProductKey"]

if str(DATA_PATH).endswith(".xlsx"):
    df = pd.read_excel(DATA_PATH, sheet_name="in")
else:
    df = pd.read_csv(DATA_PATH)

df = df[CAT_FEATURES + TEXT_FEATURES + TARGETS].copy()
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Check rows where core identifier features are completely missing
mask = df[["GPCBrickCode", "UNSPSCNumber", "ProductDescription"]].isna().all(axis=1)
print(f"Rows completely missing GPCBrickCode, UNSPSCNumber, and ProductDescription: {mask.sum()}")
df[mask].head()

In [ ]:
missing = df[CAT_FEATURES + TEXT_FEATURES].isna().sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.barh(missing.index[::-1], missing.values[::-1], color=PALETTE["categorical"][0], height=0.55)
ax.set_xlabel("Missing rows (of %d)" % len(df))
ax.set_title("Missing values by input feature (7-feature pipeline)", loc="left", fontweight="bold", color=PALETTE["ink_primary"])
ax.grid(axis="x", linewidth=0.6)
ax.set_axisbelow(True)
for b, v in zip(bars, missing.values[::-1]):
    ax.text(v + 15, b.get_y() + b.get_height()/2, f"{v} ({v/len(df):.1%})", va="center", color=PALETTE["ink_secondary"], fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Preserve baseline data filter: drop 650 unidentifiable rows
df.dropna(subset=["GPCBrickCode", "UNSPSCNumber", "ProductDescription"], how="all", inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Cleaned dataset shape: {df.shape}")
df.isna().sum()

In [ ]:
product_dfs = {
    TARGET: df[TARGET].value_counts().reset_index(name="count").sort_values(by="count")
    for TARGET in TARGETS
}

for TARGET in TARGETS[:2]:
    product_df = product_dfs[TARGET]
    plt.figure(figsize=(max(12, len(product_df) * 0.18), 6))
    ax = sns.barplot(data=product_df, x=TARGET, y="count", color=PALETTE["categorical"][0])
    for container in ax.containers:
        ax.bar_label(container, fmt="%d", padding=3, fontsize=8)
    plt.title(f"Distribution of {TARGET} classes (N={len(df)})", loc="left", fontweight="bold")
    plt.xlabel(TARGET)
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

for TARGET in TARGETS[2:]:
    counts = product_dfs[TARGET].sort_values("count", ascending=False)
    print(f"\n{TARGET}: top 10 of {len(counts)} unique classes")
    display(counts.head(10))

## 2. Fold Rare Classes into `Other`

To ensure robust cross-validation and statistical validity, any target classes with fewer than 10 occurrences are folded into the `"Other"` category (matching baseline protocol).

In [ ]:
MIN_CLASS_COUNT = 10
for TARGET in TARGETS:
    type_counts = df[TARGET].value_counts()
    rare_types = type_counts[type_counts < MIN_CLASS_COUNT].index.tolist()
    print(f"Folding {len(rare_types)} {TARGET} classes with < {MIN_CLASS_COUNT} samples into 'Other'")
    df[TARGET] = df[TARGET].where(~df[TARGET].isin(rare_types), "Other")
    print(f"{TARGET} classes: {type_counts.shape[0]} -> {df[TARGET].nunique()}\n")

## 3. Preprocessing & Stratified Train/Test Split

We apply imputations for categorical features:
- `GPCBrickCode` -> fillna `"Missing"`
- `UNSPSCNumber` -> fillna `-1` -> cast to integer -> string
- `IsMedical` -> fillna `False` -> cast to string
- `BrandName` -> fillna `"Missing"`
- `SubBrandName` -> fillna `"Missing"`

We then create an 80/20 stratified split per target (`random_state=42`), guaranteeing that the test set remains completely unseen during OneHotEncoder fitting and feature transformations.

In [ ]:
# Categorical Imputation
df["GPCBrickCode"] = df["GPCBrickCode"].fillna("Missing")
df["UNSPSCNumber"] = df["UNSPSCNumber"].fillna(-1).astype(int).astype(str)
df["IsMedical"] = df["IsMedical"].fillna(False).astype(str)
df["BrandName"] = df["BrandName"].fillna("Missing").astype(str)
df["SubBrandName"] = df["SubBrandName"].fillna("Missing").astype(str)

model_data = {}
for TARGET in TARGETS:
    df_train, df_test = train_test_split(
        df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[TARGET]
    )
    model_data[TARGET] = {
        "df_train": df_train,
        "df_test": df_test,
    }
    print(f"{TARGET}: Train = {len(df_train)} rows ({df_train[TARGET].nunique()} classes), Test = {len(df_test)} rows ({df_test[TARGET].nunique()} classes)")

## 4. One-Hot Encoding for 5 Categorical Features

We fit `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` strictly on `df_train` to prevent data leakage.

In [ ]:
for TARGET in TARGETS:
    df_train = model_data[TARGET]["df_train"]
    df_test = model_data[TARGET]["df_test"]

    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    cat_train = ohe.fit_transform(df_train[CAT_FEATURES])
    cat_test = ohe.transform(df_test[CAT_FEATURES])

    model_data[TARGET]["ohe"] = ohe
    model_data[TARGET]["cat_train"] = cat_train
    model_data[TARGET]["cat_test"] = cat_test
    print(f"{TARGET}: Categorical feature dimensions = {cat_train.shape[1]} (Train: {cat_train.shape}, Test: {cat_test.shape})")

## 5. Dense Semantic Text Embeddings (`ProductDescription` & `ProductShortName`)

We use the pretrained `SentenceTransformer('all-MiniLM-L6-v2')` model to extract 384-dimensional dense semantic representations for both `ProductDescription` and `ProductShortName`.
- Unique text values are factored and encoded in batches of 64.
- Total dense semantic dimensionality = 384 + 384 = 768 features.

In [ ]:
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
embed_dim = embedder.get_embedding_dimension()

def embed_text_column(series):
    text = series.fillna("").astype(str)
    has_text = text.str.strip() != ""
    embeddings = np.zeros((len(series), embed_dim), dtype=np.float32)
    non_empty_idx = np.where(has_text.to_numpy())[0]
    text_codes, unique_texts = pd.factorize(text[has_text], sort=False)
    if len(unique_texts):
        unique_embeddings = embedder.encode(
            unique_texts.tolist(),
            batch_size=64,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        embeddings[non_empty_idx] = unique_embeddings[text_codes]
    return embeddings, has_text.sum(), len(unique_texts)

print("Encoding ProductDescription...")
desc_embeddings, n_with_desc, n_unique_desc = embed_text_column(df["ProductDescription"])
print(f"Rows with ProductDescription: {n_with_desc} / {len(df)} (unique: {n_unique_desc})")

print("\nEncoding ProductShortName...")
short_embeddings, n_with_short, n_unique_short = embed_text_column(df["ProductShortName"])
print(f"Rows with ProductShortName: {n_with_short} / {len(df)} (unique: {n_unique_short})")

# Concatenate text features
combined_text_embeddings = np.hstack([desc_embeddings, short_embeddings])
print(f"Combined text embedding matrix shape: {combined_text_embeddings.shape}")

## 6. Combine Features into Train & Test Matrices

We horizontally stack the One-Hot Encoded categorical matrices with the dense 768-dimensional text embedding matrices.

In [ ]:
feature_shapes = {}
for TARGET in TARGETS:
    data = model_data[TARGET]
    train_pos = df.index.get_indexer(data["df_train"].index)
    test_pos = df.index.get_indexer(data["df_test"].index)

    X_train = np.hstack([data["cat_train"], combined_text_embeddings[train_pos]])
    X_test = np.hstack([data["cat_test"], combined_text_embeddings[test_pos]])
    data["X_train"] = X_train
    data["X_test"] = X_test
    data["y_train"] = data["df_train"][TARGET]
    data["y_test"] = data["df_test"][TARGET]
    feature_shapes[TARGET] = {"Train Shape": X_train.shape, "Test Shape": X_test.shape}

pd.DataFrame(feature_shapes).T

## 7. Model Training: Random Forest Classifiers (600 Trees)

We train 5 independent `RandomForestClassifier` models (600 trees, `min_samples_leaf=1`, `random_state=42`) using full parallelization (`n_jobs=-1`).

In [ ]:
for TARGET in TARGETS:
    print(f"Training 600-tree Random Forest for {TARGET}...")
    data = model_data[TARGET]
    model = RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=1,
        class_weight=None,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(data["X_train"], data["y_train"])
    data["model"] = model
    print(f"Completed training {TARGET}.")

## 8. Held-out Test Set Evaluation

We evaluate Top-1 accuracy, weighted F1 score, and complete classification reports on the held-out test set ($N=1,918$).

In [ ]:
for TARGET in TARGETS:
    data = model_data[TARGET]
    y_pred = data["model"].predict(data["X_test"])
    acc = accuracy_score(data["y_test"], y_pred)
    wf1 = f1_score(data["y_test"], y_pred, average="weighted", zero_division=0)
    data["y_pred"] = y_pred
    data["accuracy"] = acc
    data["weighted_f1"] = wf1
    print(f"\n================ {TARGET} Test Accuracy: {acc:.1%} (Weighted F1: {wf1:.4f}) ================")
    print(classification_report(data["y_test"], y_pred, zero_division=0))

## 9. Top-k Probability Evaluation (Top-1, Top-3, Top-5)

In operational master data management, human stewards review candidate recommendations. Top-k accuracy measures whether the true target class is contained within the top $k$ model candidates.

In [ ]:
def top_k_accuracy(proba, classes, y_true, k):
    class_to_idx = {c: i for i, c in enumerate(classes)}
    true_idx = y_true.map(class_to_idx).to_numpy()
    top_k_idx = np.argsort(proba, axis=1)[:, -k:]
    return np.mean([t in row for t, row in zip(true_idx, top_k_idx)])

topk_accuracy_by_target = {}
for TARGET in TARGETS:
    data = model_data[TARGET]
    proba_test = data["model"].predict_proba(data["X_test"])
    classes = data["model"].classes_
    data["proba_test"] = proba_test
    data["classes"] = classes

    topk_accuracy_by_target[TARGET] = {
        "Top-1": top_k_accuracy(proba_test, classes, data["y_test"], 1),
        "Top-3": top_k_accuracy(proba_test, classes, data["y_test"], 3),
        "Top-5": top_k_accuracy(proba_test, classes, data["y_test"], 5),
        "Weighted F1": data["weighted_f1"],
    }

topk_df = pd.DataFrame(topk_accuracy_by_target).T

ax = topk_df[["Top-1", "Top-3", "Top-5"]].plot(
    kind="bar",
    figsize=(10.5, 5.2),
    color=PALETTE["categorical"][:3],
    width=0.75,
)
ax.set_ylabel("Accuracy on Held-out Test Set")
ax.set_ylim(0, 1.1)
ax.set_title("7-Feature Pipeline: Top-k Accuracy by Taxonomy Level", loc="left", fontweight="bold", color=PALETTE["ink_primary"])
ax.grid(axis="y", linewidth=0.6)
ax.set_axisbelow(True)
ax.legend(title="Candidate Set", ncol=3)
plt.xticks(rotation=0)
for container in ax.containers:
    ax.bar_label(container, labels=[f"{value:.1%}" for value in container.datavalues], padding=2, fontsize=8)
plt.tight_layout()
plt.show()

topk_df.style.format({"Top-1": "{:.1%}", "Top-3": "{:.1%}", "Top-5": "{:.1%}", "Weighted F1": "{:.4f}"})

## 10. Head-to-Head Comparison: 3-Feature Baseline vs. 7-Feature Augmented

In [ ]:
baseline_metrics = {
    "ProductType": {"Top-1": 0.642, "Top-3": 0.889, "Top-5": 0.940, "Weighted F1": 0.620},
    "ProductGroup": {"Top-1": 0.778, "Top-3": 0.932, "Top-5": 0.965, "Weighted F1": 0.770},
    "ProductFamily": {"Top-1": 0.714, "Top-3": 0.891, "Top-5": 0.938, "Weighted F1": 0.590},
    "ProductLine": {"Top-1": 0.687, "Top-3": 0.864, "Top-5": 0.912, "Weighted F1": 0.490},
    "ProductKey": {"Top-1": 0.625, "Top-3": 0.831, "Top-5": 0.887, "Weighted F1": 0.450},
}
df_base = pd.DataFrame(baseline_metrics).T

comparison_data = []
for TARGET in TARGETS:
    comparison_data.append({
        "Target": TARGET,
        "v1.0 Top-1": df_base.loc[TARGET, "Top-1"],
        "v2.0 Top-1": topk_df.loc[TARGET, "Top-1"],
        "Delta Top-1": topk_df.loc[TARGET, "Top-1"] - df_base.loc[TARGET, "Top-1"],
        "v1.0 Top-3": df_base.loc[TARGET, "Top-3"],
        "v2.0 Top-3": topk_df.loc[TARGET, "Top-3"],
        "Delta Top-3": topk_df.loc[TARGET, "Top-3"] - df_base.loc[TARGET, "Top-3"],
        "v1.0 F1": df_base.loc[TARGET, "Weighted F1"],
        "v2.0 F1": topk_df.loc[TARGET, "Weighted F1"],
        "Delta F1": topk_df.loc[TARGET, "Weighted F1"] - df_base.loc[TARGET, "Weighted F1"],
    })

comp_df = pd.DataFrame(comparison_data).set_index("Target")

# Visual comparison of Top-1
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(TARGETS))
width = 0.35
ax.bar(x - width/2, comp_df["v1.0 Top-1"], width, label="v1.0 Baseline (3 Feat)", color=PALETTE["baseline"])
ax.bar(x + width/2, comp_df["v2.0 Top-1"], width, label="v2.0 Augmented (7 Feat)", color=PALETTE["categorical"][0])
ax.set_ylabel("Top-1 Accuracy")
ax.set_title("Head-to-Head Top-1 Accuracy: Baseline vs. Augmented Model", loc="left", fontweight="bold", color=PALETTE["ink_primary"])
ax.set_xticks(x)
ax.set_xticklabels(TARGETS)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis="y", linewidth=0.6)
for i in x:
    ax.text(i - width/2, comp_df["v1.0 Top-1"].iloc[i] + 0.02, f"{comp_df['v1.0 Top-1'].iloc[i]:.1%}", ha="center", fontsize=8)
    ax.text(i + width/2, comp_df["v2.0 Top-1"].iloc[i] + 0.02, f"{comp_df['v2.0 Top-1'].iloc[i]:.1%}", ha="center", fontsize=8, fontweight="bold")
plt.tight_layout()
plt.show()

comp_df.style.format({
    "v1.0 Top-1": "{:.1%}", "v2.0 Top-1": "{:.1%}", "Delta Top-1": "+{:.1%}",
    "v1.0 Top-3": "{:.1%}", "v2.0 Top-3": "{:.1%}", "Delta Top-3": "+{:.1%}",
    "v1.0 F1": "{:.4f}", "v2.0 F1": "{:.4f}", "Delta F1": "+{:.4f}"
})

## 11. Sample Predictions Inspection

In [ ]:
def top_k_from_proba(proba_row, classes, k=3):
    order = np.argsort(proba_row)[::-1][:k]
    return [(classes[i], proba_row[i]) for i in order]

for TARGET in TARGETS:
    data = model_data[TARGET]
    classes = data["classes"]
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(data["X_test"]), size=6, replace=False)
    print(f"\n--- Sample Predictions for {TARGET} ---")
    for i in sample_idx:
        actual = data["y_test"].to_numpy()[i]
        top3 = top_k_from_proba(data["proba_test"][i], classes, k=3)
        top3_str = ", ".join(f"{t} ({p:.0%})" for t, p in top3)
        marker = "OK" if top3[0][0] == actual else ("~~" if actual in [t for t, _ in top3] else "XX")
        print(f"[{marker}] actual={actual:<30} top3: {top3_str}")

## 12. Reusable Multi-Feature Inference Function

This function mirrors the production inference signature expected by API endpoints and the Streamlit UI.

In [ ]:
def predict_taxonomy_topk(
    gpc_brick_code,
    unspsc_number,
    is_medical,
    brand_name,
    sub_brand_name,
    product_description,
    product_short_name,
    target="ProductType",
    k=3
):
    data = model_data[target]
    gpc_val = str(gpc_brick_code) if pd.notna(gpc_brick_code) and str(gpc_brick_code).strip() else "Missing"
    try:
        unspsc_val = str(int(unspsc_number))
    except (TypeError, ValueError):
        unspsc_val = "-1"
    is_med_val = str(is_medical) if pd.notna(is_medical) else "False"
    brand_val = str(brand_name) if pd.notna(brand_name) and str(brand_name).strip() else "Missing"
    sub_brand_val = str(sub_brand_name) if pd.notna(sub_brand_name) and str(sub_brand_name).strip() else "Missing"

    input_df = pd.DataFrame([[gpc_val, unspsc_val, is_med_val, brand_val, sub_brand_val]], columns=CAT_FEATURES)
    row_cat = data["ohe"].transform(input_df)

    desc_text = str(product_description or "").strip()
    if desc_text:
        row_desc = embedder.encode([desc_text], convert_to_numpy=True)
    else:
        row_desc = np.zeros((1, embed_dim), dtype=np.float32)

    short_text = str(product_short_name or "").strip()
    if short_text:
        row_short = embedder.encode([short_text], convert_to_numpy=True)
    else:
        row_short = np.zeros((1, embed_dim), dtype=np.float32)

    row_X = np.hstack([row_cat, row_desc, row_short])
    proba_row = data["model"].predict_proba(row_X)[0]
    return top_k_from_proba(proba_row, data["model"].classes_, k=k)

# Example demonstration
example = df.iloc[5]
print("Input Sample:")
print(f"GPCBrickCode: {example['GPCBrickCode']}")
print(f"UNSPSCNumber: {example['UNSPSCNumber']}")
print(f"IsMedical: {example['IsMedical']}")
print(f"BrandName: {example['BrandName']}")
print(f"SubBrandName: {example['SubBrandName']}")
print(f"ProductShortName: {example['ProductShortName']}")
print(f"ProductDescription: {str(example['ProductDescription'])[:60]}...\n")

for TARGET in TARGETS:
    res = predict_taxonomy_topk(
        example['GPCBrickCode'],
        example['UNSPSCNumber'],
        example['IsMedical'],
        example['BrandName'],
        example['SubBrandName'],
        example['ProductDescription'],
        example['ProductShortName'],
        target=TARGET,
        k=3
    )
    print(f"{TARGET:15s} actual: {example[TARGET]:<25s} predicted: {res}")

## 13. Export Held-out Test Sets for Web UI & Validation

In [ ]:
TEST_SET_DIR = Path("test_sets")
TEST_SET_DIR.mkdir(exist_ok=True)

for TARGET in TARGETS:
    TEST_SET_PATH = TEST_SET_DIR / f"{TARGET.replace('Product', 'product_').lower()}_7feat_test_set.csv"
    df_test = model_data[TARGET]["df_test"]
    df_test[CAT_FEATURES + TEXT_FEATURES + [TARGET]].to_csv(TEST_SET_PATH, index=False)
    print(f"Saved {len(df_test)} held-out {TARGET} rows to {TEST_SET_PATH}")

## 14. Save Versioned Model Artifacts (`_rf_7feat_*`)

Serializes model classifiers, fitted encoders, and metadata dictionaries to the `artifacts/` folder.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

artifact_prefixes = {
    "ProductType": "product_type",
    "ProductGroup": "product_group",
    "ProductFamily": "product_family",
    "ProductLine": "product_line",
    "ProductKey": "product_key",
}

for TARGET in TARGETS:
    prefix = artifact_prefixes[TARGET]
    data = model_data[TARGET]
    joblib.dump(data["model"], ARTIFACT_DIR / f"{prefix}_rf_7feat_model.joblib")
    joblib.dump(data["ohe"], ARTIFACT_DIR / f"{prefix}_rf_7feat_ohe.joblib")
    joblib.dump(
        {
            "embed_model_name": EMBED_MODEL_NAME,
            "embed_dim": embed_dim,
            "cat_features": CAT_FEATURES,
            "text_features": TEXT_FEATURES,
            "classes": data["classes"].tolist(),
        },
        ARTIFACT_DIR / f"{prefix}_rf_7feat_meta.joblib",
    )
    print(f"Successfully saved {prefix} 7-feature artifacts to {ARTIFACT_DIR}")